# Neural Network Model

We will be training three types of neural network models
1. Temporal only Neural network model (Baseline)
2. Temporal + Static Neural Network (Single Stage Model)
3. Temporal Neural Network on Trend, Static Random Forest on Residuals (Two Stage Models)

This notebook illustrates the training process for the one stage neural network model on all features (Type 2). This is the neural network version of the single stage model.

The network sees only the temporal features of the month a flat was approved in (the 177 stock, COE and macroeconomic features of `data/feature_config_temporal.py`, from `data/features/final/temporal_scaled.csv` (already scaled by the data pipeline, so the network does no scaling of its own)), so every flat of a month gets the same prediction: the level of rent in that month.

Some details on the model architecture:

* **Network:**  We use an encoder like structure, with reducing hidden layer sizes to do automatic feature selection. The reducing layer size acts like a dimensionality reducer, much like PCA, with the encoding mechanism learned from gradient descent instead of computed via eigenvector decomposition.

* **Architecture** nn.ModuleList(`177 -> 64 -> 16 -> 1`), LeakyReLU activation between layers.

* **Output** The raw monthly rent as the target (standardised inside the training loop). Final layer has no activation.

* **Training Parameters** Adam Optimiser with learning rate 1e-3, batch size 2048, 100 epochs, random seed 0.

* **Sample Use:** Train on the `train` months of the final feature tables (2021-01..2024-03), evaluate on `val` months (2024-04..2025-03). Run a final fit on train + validation (2021-01..2025-03), followed by inference on all months. Returns monthly trend levels.# 02 Feed-forward network on the static and temporal features (single stage): training and inference

The network of `01_temporal_ffn.ipynb` with the static features of the flat added to its input, everything else the same: `LeakyReLU`, hidden layers `[64, 16]`, a linear output, the raw monthly rent as the target
(standardised inside the training loop), Adam with learning rate 1e-3, batch size 2048, seed 0. One network predicts the rent directly from the flat **and** the month.

* **Inputs:** the static features of the flat (`data/feature_config_static.py` from `static_train_scaled.csv` / `static_test_scaled.csv`: the scaled numeric columns, the ordinal columns standardised on the train rows and the one-hot columns; 151 inputs)
  and the 177 temporal features of the month it was approved in (`temporal_scaled.csv`, scaled by the data pipeline, as in notebook 01): 328 inputs. Identifier columns (town, block, coordinates, ...) are not inputs.
* **Rows:** one feature row per rental (149,999 train + validation rows, 50,000 test rows), so predictions are made for the original rows directly (`ROW_ID` = position in `data/original_dataset/{train,test}.csv`).
* **Number of epochs:** notebook 01's 100 epochs overfit once the flats' own features are inputs (the held-out error rises after the first few epochs), so the number of epochs is chosen on the held-out fit
  (the minimum of the validation RMSE averaged over 5 seeds, smoothed over 5 epochs) and used for both fits. The choice is made on the validation rows, which therefore slightly flatter the held-out predictions.
* **Two fits:** held-out (2021-01..2024-03, predicts the validation rows) and final (2021-01..2025-03, predicts the train rows in-sample, the test rows, and the rent of a fixed set of flats in every month).
  The growth of the market is read from **the same flats** predicted in every month: the train flats approved in 2025-01..03, predicted as if approved in each month (the flats differ between months, so the monthly mean prediction mixes growth with a change in the flat mix).
* **Outputs:** the model in `results/models/`, the training statistics in `results/tables/static_temporal_ffn/`, the predictions and the fixed-flat levels in `03_evaluation/predictions/`. Evaluation: `03_evaluation/evaluation.ipynb`.

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parents[1]))     # 02_modelling: nn_utils.py is shared by the single-stage and two-stage notebooks
import nn_utils as nu

warnings.filterwarnings('ignore')
pd.set_option('display.width', 200)
NAME = 'static_temporal_ffn'
HIDDEN = [64, 16]
ACTIVATION = 'leaky_relu'
MAX_EPOCHS, BATCH_SIZE, LEARNING_RATE, WEIGHT_DECAY, SEED = 100, 2048, 1e-3, 0.0, 0     # the same as notebook 01; the epochs actually used are chosen in section 2
SEEDS = range(5)
nu.use_results_dir(Path.cwd() / 'results')
nu.ensure_output_dirs(NAME)
fit_args = dict(hidden=HIDDEN, activation=ACTIVATION, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

## 1. Data

In [2]:
temporal = nu.load_temporal()
TEMPORAL = list(temporal.columns)
train, test, STATIC = nu.load_static()
fit, val = (train['sample_type'] == 'train').to_numpy(), (train['sample_type'] == 'val').to_numpy()
month, y = train['month'].to_numpy(), train[nu.LABEL].to_numpy()
X, X_test = train[STATIC].to_numpy(np.float32), test[STATIC].to_numpy(np.float32)

print(f'{len(STATIC)} static + {len(TEMPORAL)} temporal = {len(STATIC) + len(TEMPORAL)} inputs')
print(f'train rows {fit.sum():,} (months 0..{nu.TRAIN_LAST}), validation rows {val.sum():,} (months {nu.TRAIN_LAST + 1}..{nu.VALIDATION_LAST}), test rows {len(test):,} (months {test["month"].min()}..{test["month"].max()})')

151 static + 177 temporal = 328 inputs
train rows 114,841 (months 0..38), validation rows 35,158 (months 39..50), test rows 50,000 (months 50..66)


## 2. Held-out fit: 2021-01..2024-03, and the number of epochs

The network is trained on the train rows only for up to 100 epochs, for 5 seeds, and scored after every epoch on the validation rows (never trained on). The number of epochs is the minimum of the seed-averaged validation RMSE
(smoothed over 5 epochs). The held-out predictions are those of the seed-0 network refitted for that many epochs.

In [3]:
runs = {s: nu.fit_ffn(temporal, month[fit], y[fit], X=X[fit], valid=(month[val], X[val], y[val]), epochs=MAX_EPOCHS, seed=s, **fit_args) for s in SEEDS}
curves = pd.DataFrame({s: r['history'].set_index('epoch')['valid_rmse_dollar'] for s, r in runs.items()})
EPOCHS = int(curves.mean(axis=1).rolling(5, center=True, min_periods=1).mean().idxmin())
print(f'validation RMSE averaged over {len(SEEDS)} seeds: epoch 1 ${curves.mean(axis=1).iloc[0]:,.0f}, epoch {EPOCHS} ${curves.mean(axis=1).loc[EPOCHS]:,.0f}, epoch {MAX_EPOCHS} ${curves.mean(axis=1).iloc[-1]:,.0f}  ->  EPOCHS = {EPOCHS}')

held = nu.fit_ffn(temporal, month[fit], y[fit], X=X[fit], valid=(month[val], X[val], y[val]), epochs=EPOCHS, seed=SEED, **fit_args)
pred_val = nu.predict_ffn(held, temporal, month[val], X[val])
print(held['history'].iloc[[0, EPOCHS // 2, EPOCHS - 1]].round(1).to_string(index=False))

validation RMSE averaged over 5 seeds: epoch 1 $570, epoch 20 $522, epoch 100 $558  ->  EPOCHS = 20


 epoch  train_mse_std  train_rmse_dollar  valid_rmse_dollar
     1            0.6              586.2              556.2
    11            0.4              487.1              492.1
    20            0.4              482.4              502.5


## 3. Final fit: 2021-01..2025-03

The same network, trained for the chosen number of epochs on all the train + validation rows. The fixed flats are the train flats approved in 2025-01..03, predicted as if approved in each month 0..66.

In [4]:
final = nu.fit_ffn(temporal, month, y, X=X, epochs=EPOCHS, seed=SEED, **fit_args)
pred_train, pred_test = nu.predict_ffn(final, temporal, month, X), nu.predict_ffn(final, temporal, test['month'].to_numpy(), X_test)
level = nu.fixed_flat_levels(final, temporal, X[np.isin(month, nu.Q1_2025)], range(nu.N_MONTHS))
print(final['history'].iloc[[0, EPOCHS // 2, EPOCHS - 1]].round(1).to_string(index=False))
print(level.loc[44:].round(0).rename(lambda m: nu.month_label(m)[0]).to_frame('level of the same flats').T.to_string())

 epoch  train_mse_std  train_rmse_dollar
     1            0.6              576.1
    11            0.4              484.7
    20            0.4              481.1
                         2024-09  2024-10  2024-11  2024-12  2025-01  2025-02  2025-03  2025-04  2025-05  2025-06  2025-07  2025-08  2025-09  2025-10  2025-11  2025-12  2026-01  2026-02  2026-03  2026-04  2026-05  2026-06  2026-07
level of the same flats   3046.0   3071.0   3072.0   3042.0   3065.0   3077.0   3050.0   2936.0   3181.0   3212.0   2900.0   3250.0   3208.0   3202.0   3095.0   3110.0   2952.0   3101.0   3327.0   3340.0   3321.0   3248.0   3075.0


## 4. Save the model, the training statistics and the predictions

In [5]:
paths = [nu.MODEL_DIR / f'{NAME}.pt']
nu.save_ffn(final, paths[0], static_features=STATIC, temporal_features=TEMPORAL)
paths += [nu.save_table(runs[SEED]['history'], NAME, 'epoch_losses_heldout.csv', index=False),         # seed 0, all 100 epochs: shows where the held-out error turns
          nu.save_table(curves, NAME, 'valid_rmse_by_epoch_seeds.csv'),
          nu.save_table(held['history'], NAME, 'epoch_losses_heldout_selected.csv', index=False),        # the held-out fit used for the validation predictions (EPOCHS epochs)
          nu.save_table(final['history'], NAME, 'epoch_losses_final.csv', index=False)]
paths += [nu.save_predictions(NAME, 'val', train[val], pred_val),
          nu.save_predictions(NAME, 'train', train, pred_train),
          nu.save_predictions(NAME, 'test', test, pred_test),
          nu.save_levels(NAME, level)]
print('saved:', *paths, sep='\n  ')

saved:
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/02_modelling/single_stage_models/neural_network/results/models/static_temporal_ffn.pt
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/02_modelling/single_stage_models/neural_network/results/tables/static_temporal_ffn/epoch_losses_heldout.csv
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/02_modelling/single_stage_models/neural_network/results/tables/static_temporal_ffn/valid_rmse_by_epoch_seeds.csv
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/02_modelling/single_stage_models/neural_network/results/tables/static_temporal_ffn/epoch_losses_heldout_selected.csv
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/02_modelling/single_stage_models/neural_network/results/tables/static_temporal_ffn/epoch_losses_final.csv
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/03_evaluation/predictions/val/static_temporal_ffn.csv
  /Users/keltonn/De